# 🏠 Predicción de precios de vivienda en Bogotá

## Objetivo

En este proyecto se toma como base la tarea anterior de predicción de precios de vivienda y se personaliza para **Bogotá, Colombia**.

Se ejecutarán tres algoritmos:

1. **Regresión Lineal**: predice el precio de la vivienda.
2. **Regresión Polinomial**: busca capturar relaciones no lineales entre las características y el precio.
3. **Regresión Logística**: clasifica las viviendas en dos grupos según su precio: **precio alto** y **precio no alto**.

También se construirán cuadros y gráficos de comparación y se comentará el código para facilitar su comprensión.

> **Importante:** el dataset de Bogotá utilizado corresponde a un conjunto histórico de vivienda usada publicado en un Gist de GitHub. Por tanto, los resultados sirven para el ejercicio académico y no representan una tasación actual del mercado inmobiliario de Bogotá.


## 1. Importación de librerías

En esta sección importamos las herramientas necesarias para cargar, organizar, visualizar y modelar los datos.


In [ ]:
# Importamos pandas para manipular los datos en forma de tablas.
import pandas as pd

# Importamos numpy para realizar operaciones numéricas.
import numpy as np

# Importamos matplotlib para crear gráficos.
import matplotlib.pyplot as plt

# Importamos seaborn para crear gráficos estadísticos.
import seaborn as sns

# Importamos train_test_split para separar los datos de entrenamiento y prueba.
from sklearn.model_selection import train_test_split

# Importamos LinearRegression para construir el modelo de regresión lineal.
from sklearn.linear_model import LinearRegression

# Importamos PolynomialFeatures para transformar las variables y crear términos polinomiales.
from sklearn.preprocessing import PolynomialFeatures

# Importamos LogisticRegression para clasificar las viviendas según su nivel de precio.
from sklearn.linear_model import LogisticRegression

# Importamos métricas para evaluar los modelos de regresión.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importamos métricas para evaluar el modelo de clasificación.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Importamos Pipeline para unir la transformación polinomial con el modelo lineal.
from sklearn.pipeline import make_pipeline


## 2. Cargar el dataset de vivienda usada

El archivo contiene información de diferentes viviendas de Colombia. Primero lo cargamos y después filtramos únicamente los registros correspondientes a **Bogotá**.


In [ ]:
# Guardamos en una variable la dirección del dataset utilizado para el proyecto.
url_dataset = "https://gist.githubusercontent.com/carojasb/5668dbb6af4dce0db91158c63e13944a/raw/31e47931fa022214c42610cdd10023cadf73aeb3/gistfile1.txt"

# Leemos el archivo CSV directamente desde Internet.
df = pd.read_csv(url_dataset)

# Mostramos las primeras cinco filas para conocer la estructura de los datos.
df.head()


In [ ]:
# Mostramos el número de filas y columnas disponibles en el dataset original.
print("Dimensiones del dataset original:", df.shape)

# Mostramos los nombres de todas las columnas disponibles.
print("\nColumnas del dataset:")
print(df.columns.tolist())


## 3. Personalizar los datos para Bogotá

El dataset contiene información de diferentes departamentos y ciudades. Para esta tarea nos quedaremos únicamente con las viviendas cuyo departamento sea Bogotá.


In [ ]:
# Filtramos las filas cuyo departamento corresponde a Bogotá.
df_bogota = df[df["DEPARTAMENTO"].astype(str).str.upper().str.strip() == "BOGOTÁ"].copy()

# Mostramos cuántos registros quedaron después del filtro.
print("Cantidad de viviendas de Bogotá:", len(df_bogota))

# Mostramos las primeras viviendas seleccionadas.
df_bogota.head()


## 4. Seleccionar las variables que utilizaremos

Para que los algoritmos trabajen correctamente utilizaremos variables numéricas que tienen una relación directa con el precio:

- `AREA_DESDE`: área de la vivienda en metros cuadrados.
- `NUMERO_HABITACIONES`: número de habitaciones.
- `NUMERO_BAÑOS`: número de baños.
- `PRECIO_DESDE`: precio de la vivienda, que será nuestra variable objetivo.

No utilizaremos las columnas de texto como barrio, dirección o descripción en estos tres modelos básicos.


In [ ]:
# Seleccionamos únicamente las cuatro columnas necesarias para el ejercicio.
datos = df_bogota[[
    "PRECIO_DESDE",
    "AREA_DESDE",
    "NUMERO_HABITACIONES",
    "NUMERO_BAÑOS"
]].copy()

# Convertimos las columnas numéricas a valores numéricos.
# Los valores que no puedan convertirse se transforman en valores nulos.
for columna in datos.columns:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

# Eliminamos las filas que tengan valores nulos en las variables utilizadas.
datos = datos.dropna()

# Eliminamos precios y áreas que sean menores o iguales a cero porque no son válidos para este análisis.
datos = datos[(datos["PRECIO_DESDE"] > 0) & (datos["AREA_DESDE"] > 0)]

# Mostramos las primeras filas del conjunto limpio.
datos.head()


In [ ]:
# Mostramos información general del conjunto de datos limpio.
datos.info()

# Mostramos estadísticas descriptivas de las variables numéricas.
datos.describe()


## 5. Crear el precio por metro cuadrado

Esta variable nos permite personalizar todavía más el análisis para Bogotá, ya que relaciona el precio de cada vivienda con su área.


In [ ]:
# Calculamos el precio por metro cuadrado dividiendo el precio entre el área.
datos["PRECIO_M2"] = datos["PRECIO_DESDE"] / datos["AREA_DESDE"]

# Mostramos las primeras filas incluyendo el nuevo indicador.
datos.head()


## 6. Visualización inicial

Antes de entrenar los modelos observamos la relación entre el área y el precio de las viviendas.


In [ ]:
# Creamos una figura con un tamaño adecuado para el gráfico.
plt.figure(figsize=(10, 6))

# Creamos un gráfico de dispersión entre el área y el precio.
sns.scatterplot(data=datos, x="AREA_DESDE", y="PRECIO_DESDE", alpha=0.6)

# Agregamos un título al gráfico.
plt.title("Área de la vivienda vs. precio en Bogotá")

# Nombramos el eje X.
plt.xlabel("Área (m²)")

# Nombramos el eje Y.
plt.ylabel("Precio desde (COP)")

# Mostramos el gráfico.
plt.show()


# 7. Preparar los datos para los modelos de regresión

La variable `X` contiene las características que utilizaremos para predecir el precio.

La variable `y` contiene el precio que queremos predecir.


In [ ]:
# Seleccionamos las variables predictoras.
X = datos[[
    "AREA_DESDE",
    "NUMERO_HABITACIONES",
    "NUMERO_BAÑOS"
]]

# Seleccionamos el precio como variable objetivo.
y = datos["PRECIO_DESDE"]

# Dividimos los datos en entrenamiento y prueba.
# El 80 % se utiliza para entrenar y el 20 % para evaluar.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Mostramos el tamaño de los conjuntos.
print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)


# 8. Regresión Lineal

La Regresión Lineal intenta encontrar una relación lineal entre las características de una vivienda y su precio.


In [ ]:
# Creamos el modelo de regresión lineal.
modelo_lineal = LinearRegression()

# Entrenamos el modelo con los datos de entrenamiento.
modelo_lineal.fit(X_train, y_train)

# Generamos las predicciones sobre los datos de prueba.
pred_lineal = modelo_lineal.predict(X_test)

# Calculamos el error absoluto medio.
mae_lineal = mean_absolute_error(y_test, pred_lineal)

# Calculamos el error cuadrático medio.
mse_lineal = mean_squared_error(y_test, pred_lineal)

# Calculamos la raíz del error cuadrático medio.
rmse_lineal = np.sqrt(mse_lineal)

# Calculamos el coeficiente R².
r2_lineal = r2_score(y_test, pred_lineal)

# Mostramos las métricas del modelo lineal.
print(f"MAE Regresión Lineal: ${mae_lineal:,.0f}")
print(f"RMSE Regresión Lineal: ${rmse_lineal:,.0f}")
print(f"R² Regresión Lineal: {r2_lineal:.4f}")


# 9. Regresión Polinomial

La regresión polinomial permite representar relaciones curvas. Primero transformamos las variables originales en términos polinomiales y después utilizamos una regresión lineal sobre esas nuevas variables.


In [ ]:
# Creamos un modelo polinomial de grado 2 unido a una regresión lineal.
modelo_polinomial = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)

# Entrenamos el modelo polinomial con los datos de entrenamiento.
modelo_polinomial.fit(X_train, y_train)

# Generamos las predicciones para los datos de prueba.
pred_polinomial = modelo_polinomial.predict(X_test)

# Calculamos el error absoluto medio del modelo polinomial.
mae_polinomial = mean_absolute_error(y_test, pred_polinomial)

# Calculamos el error cuadrático medio del modelo polinomial.
mse_polinomial = mean_squared_error(y_test, pred_polinomial)

# Calculamos la raíz del error cuadrático medio del modelo polinomial.
rmse_polinomial = np.sqrt(mse_polinomial)

# Calculamos el R² del modelo polinomial.
r2_polinomial = r2_score(y_test, pred_polinomial)

# Mostramos las métricas del modelo polinomial.
print(f"MAE Regresión Polinomial: ${mae_polinomial:,.0f}")
print(f"RMSE Regresión Polinomial: ${rmse_polinomial:,.0f}")
print(f"R² Regresión Polinomial: {r2_polinomial:.4f}")


# 10. Comparación de los modelos de regresión

En esta tabla comparamos Regresión Lineal y Regresión Polinomial.

Para MAE y RMSE, **un valor menor es mejor**.

Para R², **un valor más cercano a 1 es mejor**.


In [ ]:
# Creamos una tabla con las métricas de los dos modelos de regresión.
comparacion_regresion = pd.DataFrame({
    "Modelo": [
        "Regresión Lineal",
        "Regresión Polinomial"
    ],
    "MAE": [
        mae_lineal,
        mae_polinomial
    ],
    "RMSE": [
        rmse_lineal,
        rmse_polinomial
    ],
    "R2": [
        r2_lineal,
        r2_polinomial
    ]
})

# Mostramos la tabla de comparación.
comparacion_regresion


In [ ]:
# Creamos una figura para comparar el R² de los modelos.
plt.figure(figsize=(8, 5))

# Dibujamos las barras correspondientes al R².
sns.barplot(data=comparacion_regresion, x="Modelo", y="R2")

# Añadimos un título.
plt.title("Comparación del R²: modelos de regresión")

# Etiquetamos el eje X.
plt.xlabel("Modelo")

# Etiquetamos el eje Y.
plt.ylabel("R²")

# Mostramos el gráfico.
plt.show()


In [ ]:
# Creamos una figura para comparar los errores RMSE.
plt.figure(figsize=(8, 5))

# Dibujamos las barras correspondientes al RMSE.
sns.barplot(data=comparacion_regresion, x="Modelo", y="RMSE")

# Añadimos un título.
plt.title("Comparación del RMSE: modelos de regresión")

# Etiquetamos el eje X.
plt.xlabel("Modelo")

# Etiquetamos el eje Y.
plt.ylabel("RMSE en COP")

# Mostramos el gráfico.
plt.show()


# 11. Regresión Logística

La Regresión Logística **no predice directamente un precio continuo**. Por eso debemos convertir el problema en una clasificación.

Crearemos una nueva variable llamada `PRECIO_ALTO`:

- `1` = vivienda con precio igual o superior a la mediana.
- `0` = vivienda con precio inferior a la mediana.

De esta manera la Regresión Logística aprenderá a clasificar una vivienda como de **precio alto** o **precio no alto**.


In [ ]:
# Calculamos la mediana de los precios de las viviendas de Bogotá.
mediana_precio = datos["PRECIO_DESDE"].median()

# Creamos una variable binaria: 1 si el precio está por encima o igual a la mediana y 0 en caso contrario.
datos["PRECIO_ALTO"] = (datos["PRECIO_DESDE"] >= mediana_precio).astype(int)

# Mostramos la mediana utilizada como punto de corte.
print(f"Mediana del precio: ${mediana_precio:,.0f}")

# Mostramos cuántas viviendas pertenecen a cada categoría.
print("\nCantidad de viviendas por categoría:")
print(datos["PRECIO_ALTO"].value_counts())


In [ ]:
# Seleccionamos las mismas características utilizadas en las regresiones.
X_log = datos[[
    "AREA_DESDE",
    "NUMERO_HABITACIONES",
    "NUMERO_BAÑOS"
]]

# Seleccionamos la nueva variable de clasificación.
y_log = datos["PRECIO_ALTO"]

# Dividimos los datos de clasificación en entrenamiento y prueba.
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log,
    y_log,
    test_size=0.20,
    random_state=42,
    stratify=y_log
)

# Creamos el modelo de Regresión Logística.
modelo_logistico = LogisticRegression(max_iter=1000)

# Entrenamos el modelo logístico.
modelo_logistico.fit(X_train_log, y_train_log)

# Generamos las predicciones de clasificación.
pred_logistico = modelo_logistico.predict(X_test_log)

# Calculamos la exactitud del modelo.
accuracy_log = accuracy_score(y_test_log, pred_logistico)

# Calculamos la precisión del modelo.
precision_log = precision_score(y_test_log, pred_logistico, zero_division=0)

# Calculamos el recall del modelo.
recall_log = recall_score(y_test_log, pred_logistico, zero_division=0)

# Calculamos el F1-score del modelo.
f1_log = f1_score(y_test_log, pred_logistico, zero_division=0)

# Mostramos las métricas de clasificación.
print(f"Accuracy: {accuracy_log:.4f}")
print(f"Precision: {precision_log:.4f}")
print(f"Recall: {recall_log:.4f}")
print(f"F1-score: {f1_log:.4f}")


## 12. Matriz de confusión

La matriz de confusión permite observar cuántas viviendas fueron clasificadas correctamente y cuántas fueron clasificadas de manera incorrecta.


In [ ]:
# Calculamos la matriz de confusión.
matriz = confusion_matrix(y_test_log, pred_logistico)

# Creamos una figura para visualizar la matriz.
plt.figure(figsize=(6, 5))

# Mostramos la matriz como un mapa de calor.
sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Precio no alto", "Precio alto"],
    yticklabels=["Precio no alto", "Precio alto"]
)

# Añadimos el título.
plt.title("Matriz de confusión - Regresión Logística")

# Etiquetamos el eje X.
plt.xlabel("Predicción")

# Etiquetamos el eje Y.
plt.ylabel("Valor real")

# Mostramos el gráfico.
plt.show()


# 13. Cuadro final de comparación

Aquí reunimos los resultados principales.

Como la Regresión Logística es un problema de **clasificación**, no se debe comparar su RMSE o R² directamente con los modelos de regresión. Por eso se presentan métricas adecuadas para cada tipo de problema.


In [ ]:
# Creamos una tabla para comparar los principales resultados de los tres algoritmos.
comparacion_final = pd.DataFrame({
    "Algoritmo": [
        "Regresión Lineal",
        "Regresión Polinomial",
        "Regresión Logística"
    ],
    "Tipo de problema": [
        "Regresión",
        "Regresión",
        "Clasificación"
    ],
    "Métrica principal": [
        "R²",
        "R²",
        "Accuracy"
    ],
    "Resultado": [
        r2_lineal,
        r2_polinomial,
        accuracy_log
    ]
})

# Mostramos la tabla final.
comparacion_final


In [ ]:
# Creamos un gráfico con los resultados principales de cada algoritmo.
plt.figure(figsize=(10, 6))

# Dibujamos una barra para cada algoritmo.
sns.barplot(data=comparacion_final, x="Algoritmo", y="Resultado")

# Añadimos un título.
plt.title("Comparación de los algoritmos")

# Etiquetamos el eje X.
plt.xlabel("Algoritmo")

# Etiquetamos el eje Y.
plt.ylabel("Resultado de la métrica principal")

# Ajustamos los elementos del gráfico para evitar cortes.
plt.tight_layout()

# Mostramos el gráfico.
plt.show()


# 14. Predicción de una vivienda de Bogotá

Finalmente, utilizamos los modelos de regresión para estimar el precio de una vivienda hipotética con características definidas por nosotros.


In [ ]:
# Creamos una vivienda hipotética de Bogotá.
nueva_vivienda = pd.DataFrame({
    "AREA_DESDE": [70],
    "NUMERO_HABITACIONES": [3],
    "NUMERO_BAÑOS": [2]
})

# Utilizamos la Regresión Lineal para estimar su precio.
precio_lineal_nuevo = modelo_lineal.predict(nueva_vivienda)[0]

# Utilizamos la Regresión Polinomial para estimar su precio.
precio_polinomial_nuevo = modelo_polinomial.predict(nueva_vivienda)[0]

# Mostramos la predicción de la Regresión Lineal.
print(f"Precio estimado por Regresión Lineal: ${precio_lineal_nuevo:,.0f} COP")

# Mostramos la predicción de la Regresión Polinomial.
print(f"Precio estimado por Regresión Polinomial: ${precio_polinomial_nuevo:,.0f} COP")

# Utilizamos la Regresión Logística para clasificar la vivienda.
clasificacion_nueva = modelo_logistico.predict(nueva_vivienda)[0]

# Convertimos el resultado numérico a una explicación entendible.
resultado_clasificacion = "Precio alto" if clasificacion_nueva == 1 else "Precio no alto"

# Mostramos la clasificación de la vivienda.
print(f"Clasificación logística: {resultado_clasificacion}")


# 15. Conclusiones

En este proyecto se tomó como base el ejercicio anterior de predicción de precios y se adaptó a datos de viviendas de Bogotá.

La **Regresión Lineal** permite estimar directamente el precio de una vivienda a partir de su área, número de habitaciones y número de baños.

La **Regresión Polinomial** amplía la capacidad del modelo al permitir relaciones no lineales entre las características de la vivienda y el precio.

La **Regresión Logística** se utilizó con un objetivo diferente: clasificar las viviendas entre precio alto y precio no alto utilizando como punto de corte la mediana del precio.

Para comparar los modelos de regresión se utilizaron MAE, RMSE y R². Para la clasificación se utilizaron Accuracy, Precision, Recall y F1-score.

El modelo con mayor R² entre los modelos de regresión representa el mejor ajuste según esa métrica, mientras que en clasificación una mayor Accuracy indica un mejor porcentaje de clasificaciones correctas.

**Nota:** el dataset utilizado es histórico y las predicciones son académicas. No deben utilizarse como valoración comercial actual de inmuebles en Bogotá.
